# Creation of Figures

## Step 1: Importing Libraries
All required libraries and classes are imported below.

In [1]:
# Libraries
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Plotting
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from kaleido import get_chrome_sync
get_chrome_sync()

## Step 2: Loading Feature DataFrame
In the following cell, the DataFrame `features.parquet`, which was created in the notebook `features_vX.ipynb` and contains all features for our ML model, is imported.

#### <font color='red'> CHANGE THE START AND END TIMES OF THE ANALYSES IN THE FIRST LINES OF CODE: </font>

In [2]:
# ========================= Adjust the parameters here inside =========================

START = ["2025-01-01 00:00"]  # Starting timestamp for the filtering.
STOP = ["2026-01-01 00:00"]   # Ending timestamp for the filtering.

# ========================= Adjust the parameters here inside =========================


# Load the features data.
features_df = pd.read_parquet("data/processed/features.parquet")

# Convert the taxi_start column to datetime if it is not already in that format.
features_df["taxi_start"] = pd.to_datetime(
    features_df["taxi_start"],
    utc=True
)

# Filter the dataset by START and STOP.
filtered_dfs = []

for i in range(len(START)):
    start_ts = pd.Timestamp(START[i], tz="UTC")
    stop_ts = pd.Timestamp(STOP[i], tz="UTC")

    period_df = features_df.loc[
        (features_df["taxi_start"] >= start_ts)
        & (features_df["taxi_start"] < stop_ts)
    ]

    filtered_dfs.append(period_df)

# Combine all selected periods.
features_df = pd.concat(
    filtered_dfs,
    ignore_index=True
)

In [3]:
features_df.tail()

In [4]:
runways = ["28", "32", "16", "10", "34"]

for runway in runways:
    count = features_df[f"TO_runway_{runway}"].sum()
    print(f"Runway {runway}: {count:.0f} departures")

## Step 3: Creating Figures
In the following cells, figures are created to support our thesis and provide an overview of the taxi-out situation at ZRH with regard to the available features/data. These figures are saved as PDF files in the `figures` folder.

In [ ]:
# ========================= Adjust the parameters here inside =========================

RUNWAYS = ["28", "32", "16", "10", "34"]  # Runway order in the plot.

TAXI_TIME_MIN = -0.4                      # [min] Lower x-axis limit.
TAXI_TIME_MAX = 45.4                      # [min] Upper x-axis limit.
VIOLIN_WIDTH = 0.65                       # Width of each violin.
LINE_HALF_HEIGHT = 0.4                    # Half-height of the percentile lines per runway.

# --- Parameters for fonts ---
font_family = "Arial"
font_colour = "black"

title_size = 35
axis_title_size = 25
axis_tick_size = 25
legend_title_size = 30
legend_font_size = 25

# --- Figure size ---
figure_width = 1500
figure_height = 1100

# --- Colours ---
violin_colour = "navy"
q10_colour = "green"
median_colour = "darkorange"
q90_colour = "red"

# ========================= Adjust the parameters here inside =========================


df = features_df.copy()

# Keep only rows with valid taxi-out times.
df = df[df["taxi_time_min"].notna()]
df = df[df["taxi_time_min"] >= 0]

# Remove taxi-time outliers outside the plotted x-axis range.
df = df[
    (df["taxi_time_min"] >= TAXI_TIME_MIN)
    & (df["taxi_time_min"] <= TAXI_TIME_MAX)
].copy()


# ========================= Prepare data =========================

runway_dfs = []

for runway in RUNWAYS:
    runway_col = f"TO_runway_{runway}"

    if runway_col not in df.columns:
        print(f"Column {runway_col} not found.")
        continue

    tmp = df[df[runway_col] == 1].copy()
    tmp["runway"] = runway
    tmp["runway_idx"] = RUNWAYS.index(runway)

    runway_dfs.append(tmp)

plot_df = pd.concat(runway_dfs, ignore_index=True)

# Compute percentiles per runway.
percentiles = (
    plot_df
    .groupby(["runway", "runway_idx"])["taxi_time_min"]
    .quantile([0.10, 0.50, 0.90])
    .unstack()
    .reset_index()
    .rename(
        columns={
            0.10: "q10",
            0.50: "median",
            0.90: "q90",
        }
    )
)


# ========================= Figure =========================

fig = go.Figure()

# Add one horizontal violin per runway.
for runway in RUNWAYS:
    runway_data = plot_df[plot_df["runway"] == runway]
    runway_idx = RUNWAYS.index(runway)

    fig.add_trace(
        go.Violin(
            x=runway_data["taxi_time_min"],
            y0=runway_idx,
            orientation="h",
            width=VIOLIN_WIDTH,
            spanmode="hard",
            line=dict(
                color=violin_colour,
                width=2,
            ),
            fillcolor=violin_colour,
            opacity=0.75,
            points=False,
            meanline_visible=False,
            showlegend=False,
            hovertemplate=(
                f"Runway: {runway}<br>"
                "Taxi-out time: %{x:.2f} min"
                "<extra></extra>"
            ),
        )
    )

# Add dashed percentile lines per runway.
for _, row in percentiles.iterrows():
    runway = row["runway"]
    runway_idx = row["runway_idx"]

    y0 = runway_idx - LINE_HALF_HEIGHT
    y1 = runway_idx + LINE_HALF_HEIGHT

    fig.add_trace(
        go.Scatter(
            x=[row["q10"], row["q10"]],
            y=[y0, y1],
            mode="lines",
            line=dict(
                color=q10_colour,
                width=4,
                dash="dash",
            ),
            showlegend=False,
            hovertemplate=(
                f"Runway: {runway}<br>"
                f"10th Percentile: {row['q10']:.2f} min"
                "<extra></extra>"
            ),
        )
    )

    fig.add_trace(
        go.Scatter(
            x=[row["median"], row["median"]],
            y=[y0, y1],
            mode="lines",
            line=dict(
                color=median_colour,
                width=4,
                dash="dash",
            ),
            showlegend=False,
            hovertemplate=(
                f"Runway: {runway}<br>"
                f"Median: {row['median']:.2f} min"
                "<extra></extra>"
            ),
        )
    )

    fig.add_trace(
        go.Scatter(
            x=[row["q90"], row["q90"]],
            y=[y0, y1],
            mode="lines",
            line=dict(
                color=q90_colour,
                width=4,
                dash="dash",
            ),
            showlegend=False,
            hovertemplate=(
                f"Runway: {runway}<br>"
                f"90th Percentile: {row['q90']:.2f} min"
                "<extra></extra>"
            ),
        )
    )

# Add legend entries for the percentile lines.
fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        line=dict(
            color=q10_colour,
            width=3,
            dash="dash",
        ),
        showlegend=True,
        name="10th Percentile",
    )
)

fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        line=dict(
            color=median_colour,
            width=3,
            dash="dash",
        ),
        showlegend=True,
        name="Median",
    )
)

fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        line=dict(
            color=q90_colour,
            width=3,
            dash="dash",
        ),
        showlegend=True,
        name="90th Percentile",
    )
)

fig.update_layout(
    title=dict(
        text="Distribution of Taxi-Out Times by Runway",
        x=0.5,
        xanchor="center",
        y=0.97,
        yanchor="top",
    ),
    title_font=dict(
        size=title_size,
        family=font_family,
        color=font_colour,
    ),
    width=figure_width,
    height=figure_height,
    margin=dict(t=120, b=100, l=100),
    font=dict(
        family=font_family,
        color=font_colour,
    ),
    legend=dict(
        title="Legend",
        x=0.987,
        y=0.02,
        xanchor="right",
        yanchor="bottom",
        font=dict(
            size=legend_font_size,
            family=font_family,
            color=font_colour,
        ),
        title_font=dict(
            size=legend_title_size,
            family=font_family,
            color=font_colour,
        ),
    ),
    xaxis=dict(
        title="Taxi-Out Time [min]",
        range=[TAXI_TIME_MIN, TAXI_TIME_MAX],
        dtick=5,
        title_font=dict(
            size=axis_title_size,
            family=font_family,
            color=font_colour,
        ),
        tickfont=dict(
            size=axis_tick_size,
            family=font_family,
            color=font_colour,
        ),
    ),
    yaxis=dict(
        title="Take-Off Runway",
        tickmode="array",
        tickvals=list(range(len(RUNWAYS))),
        ticktext=RUNWAYS,
        autorange="reversed",
        title_font=dict(
            size=axis_title_size,
            family=font_family,
            color=font_colour,
        ),
        tickfont=dict(
            size=axis_tick_size,
            family=font_family,
            color=font_colour,
        ),
    ),
)

fig.show()
fig.write_image("figures/taxi_out_time_distribution_by_runway.pdf")

In [5]:
# ========================= Adjust the parameters here inside =========================

RUNWAYS = ["28", "32", "16", "10", "34"]  # Runway order in the plot.

TAXI_TIME_MIN = -1                        # [min] Lower y-axis limit.
TAXI_TIME_MAX = 46                        # [min] Upper y-axis limit.

TIME_TICK_STEP_HOURS = 1                  # X-axis tick spacing in hours.

MARKER_SIZE = 7                           # Scatter marker size in the plot.
LEGEND_MARKER_SIZE = 11                   # Scatter marker size in the legend.
MARKER_OPACITY = 0.65                     # Scatter marker opacity.

# --- Parameters for fonts ---
font_family = "Arial"
font_colour = "black"

title_size = 35
axis_title_size = 25
axis_tick_size = 25
legend_title_size = 30
legend_font_size = 25

# --- Figure size ---
figure_width = 1500
figure_height = 900

# --- Colours ---
runway_colours = {
    "28": "royalblue",
    "32": "darkred",
    "16": "darkorange",
    "10": "darkgreen",
    "34": "darkviolet",
}

# ========================= Adjust the parameters here inside =========================


df = features_df.copy()

# Convert taxi start time from UTC to Swiss local time.
df["taxi_start"] = pd.to_datetime(df["taxi_start"], utc=True)
df["taxi_start_lt"] = df["taxi_start"].dt.tz_convert("Europe/Zurich")

# Keep only rows with valid taxi start times and taxi-out times.
df = df.dropna(
    subset=[
        "taxi_start_lt",
        "taxi_time_min",
    ]
)

# Keep only rows with non-negative taxi times.
df = df[df["taxi_time_min"] >= 0]

# Remove taxi-time outliers outside the plotted y-axis range.
df = df[
    (df["taxi_time_min"] >= TAXI_TIME_MIN)
    & (df["taxi_time_min"] <= TAXI_TIME_MAX)
].copy()

# Convert local time of day to minutes after midnight.
df["time_of_day_min"] = (
    df["taxi_start_lt"].dt.hour * 60
    + df["taxi_start_lt"].dt.minute
    + df["taxi_start_lt"].dt.second / 60.0
)

# Create local-time labels for hover information.
df["time_of_day_label"] = df["taxi_start_lt"].dt.strftime("%H:%M:%S")
df["taxi_start_lt_label"] = df["taxi_start_lt"].dt.strftime("%Y-%m-%d %H:%M:%S %Z")


# ========================= Prepare data =========================

runway_dfs = []

for runway in RUNWAYS:
    runway_col = f"TO_runway_{runway}"

    tmp = df[df[runway_col] == 1].copy()
    tmp["runway"] = runway

    runway_dfs.append(tmp)

plot_df = pd.concat(runway_dfs, ignore_index=True)


# ========================= Figure =========================

fig = go.Figure()

# Add one scatter trace per runway.
for runway in RUNWAYS:
    runway_data = plot_df[plot_df["runway"] == runway]

    fig.add_trace(
        go.Scatter(
            x=runway_data["time_of_day_min"],
            y=runway_data["taxi_time_min"],
            mode="markers",
            name=runway,
            showlegend=False,
            marker=dict(
                size=MARKER_SIZE,
                opacity=MARKER_OPACITY,
                color=runway_colours[runway],
                line=dict(
                    width=0,
                ),
            ),
            customdata=runway_data[
                [
                    "time_of_day_label",
                    "taxi_start_lt_label",
                ]
            ].to_numpy(),
            hovertemplate=(
                f"Runway: {runway}<br>"
                "Local time: %{customdata[0]}<br>"
                "Taxi-out time: %{y:.2f} min"
                "<extra></extra>"
            ),
        )
    )

# Add separate dummy traces for the legend.
for runway in RUNWAYS:
    fig.add_trace(
        go.Scatter(
            x=[None],
            y=[None],
            mode="markers",
            name=runway,
            showlegend=True,
            marker=dict(
                size=LEGEND_MARKER_SIZE,
                opacity=MARKER_OPACITY,
                color=runway_colours[runway],
                line=dict(
                    width=0,
                ),
            ),
            hoverinfo="skip",
        )
    )

# X-axis ticks from 00:00 to 24:00.
tickvals = list(range(0, 24 * 60 + 1, TIME_TICK_STEP_HOURS * 60))
ticktext = [f"{h:02d}:00" for h in range(0, 25, TIME_TICK_STEP_HOURS)]

fig.update_layout(
    title=dict(
        text="Taxi-Out Time by Time of Day and Take-Off Runway",
        x=0.5,
        xanchor="center",
        y=0.97,
        yanchor="top",
    ),
    title_font=dict(
        size=title_size,
        family=font_family,
        color=font_colour,
    ),
    width=figure_width,
    height=figure_height,
    margin=dict(t=120, b=100, l=100),
    font=dict(
        family=font_family,
        color=font_colour,
    ),
    legend=dict(
        title="Take-Off Runway",
        x=0.015,
        y=0.974,
        xanchor="left",
        yanchor="top",
        font=dict(
            size=legend_font_size,
            family=font_family,
            color=font_colour,
        ),
        title_font=dict(
            size=legend_title_size,
            family=font_family,
            color=font_colour,
        ),
    ),
    xaxis=dict(
        title="Time of Day (Local Time)",
        range=[0, 24 * 60],
        tickmode="array",
        tickvals=tickvals,
        ticktext=ticktext,
        tickangle=90,
        title_font=dict(
            size=axis_title_size,
            family=font_family,
            color=font_colour,
        ),
        tickfont=dict(
            size=axis_tick_size,
            family=font_family,
            color=font_colour,
        ),
        showgrid=True,
    ),
    yaxis=dict(
        title="Taxi-Out Time [min]",
        range=[TAXI_TIME_MIN, TAXI_TIME_MAX],
        title_font=dict(
            size=axis_title_size,
            family=font_family,
            color=font_colour,
        ),
        tickfont=dict(
            size=axis_tick_size,
            family=font_family,
            color=font_colour,
        ),
        showgrid=True,
    ),
)

fig.show()
fig.write_image("figures/taxi_out_time_by_time_of_day.pdf")

In [6]:
# ========================= Adjust the parameters here inside =========================

RUNWAYS = ["28", "32", "16", "10", "34"]  # Runway order in the plot.

TAXI_TIME_MIN = -1                        # [min] Lower y-axis limit.
TAXI_TIME_MAX = 46                        # [min] Upper y-axis limit.

MOVEMENT_MIN = 0                          # Lower x-axis limit for movements.
MOVEMENT_MAX = None                       # Upper x-axis limit for movements. Use None for automatic scaling.

TIME_TICK_STEP_HOURS = 1                  # X-axis tick spacing in hours.

MARKER_SIZE = 7                           # Scatter marker size in the plot.
LEGEND_MARKER_SIZE = 11                   # Scatter marker size in the legend.
MARKER_OPACITY = 0.65                     # Scatter marker opacity.

LINE_WIDTH = 3                            # Line width in the upper subplot.
LINE_MARKER_SIZE = 8                      # Marker size for the line plot.

# --- Parameters for fonts ---
font_family = "Arial"
font_colour = "black"

title_size = 35
axis_title_size = 25
axis_tick_size = 25
legend_title_size = 30
legend_font_size = 25
subplot_title_size = 30

# --- Figure size ---
figure_width = 1500
figure_height = 1200

# --- Colours ---
runway_colours = {
    "28": "royalblue",
    "32": "darkred",
    "16": "darkorange",
    "10": "darkgreen",
    "34": "darkviolet",
}

# ========================= Adjust the parameters here inside =========================


df = features_df.copy()

# Convert taxi start time from UTC to Swiss local time.
df["taxi_start"] = pd.to_datetime(df["taxi_start"], utc=True)
df["taxi_start_lt"] = df["taxi_start"].dt.tz_convert("Europe/Zurich")

# Extract the local date.
df["date_lt"] = df["taxi_start_lt"].dt.date

# Keep only rows with valid taxi start times and taxi-out times.
df = df.dropna(
    subset=[
        "taxi_start_lt",
        "taxi_time_min",
    ]
)

# Keep only rows with non-negative taxi times.
df = df[df["taxi_time_min"] >= 0]

# Remove taxi-time outliers outside the plotted y-axis range.
df = df[
    (df["taxi_time_min"] >= TAXI_TIME_MIN)
    & (df["taxi_time_min"] <= TAXI_TIME_MAX)
].copy()

# Convert local time of day to 30-minute slots.
df["time_slot"] = (
    df["taxi_start_lt"].dt.hour * 60
    + df["taxi_start_lt"].dt.minute
) // 30


def slot_label(slot: int) -> str:
    """Return a time interval label for a given 30-minute slot index."""
    start_min = slot * 30
    end_min = start_min + 29
    h1, m1 = divmod(start_min, 60)
    h2, m2 = divmod(end_min, 60)
    return f"{h1:02d}:{m1:02d}–{h2:02d}:{m2:02d}"


# Create local-time labels for hover information.
df["time_label"] = df["time_slot"].apply(slot_label)
df["time_of_day_label"] = df["taxi_start_lt"].dt.strftime("%H:%M:%S")
df["taxi_start_lt_label"] = df["taxi_start_lt"].dt.strftime("%Y-%m-%d %H:%M:%S %Z")


# ========================= Prepare data =========================

runway_dfs = []

for runway in RUNWAYS:
    runway_col = f"TO_runway_{runway}"

    tmp = df[df[runway_col] == 1].copy()
    tmp["runway"] = runway

    runway_dfs.append(tmp)

plot_df = pd.concat(runway_dfs, ignore_index=True)

# Define the full ordered list of 30-minute slots.
time_order = [slot_label(i) for i in range(48)]

all_slots = pd.DataFrame({
    "time_slot": range(48),
    "time_label": time_order,
})

all_runway_slots = (
    pd.MultiIndex
    .from_product([RUNWAYS, range(48)], names=["runway", "time_slot"])
    .to_frame(index=False)
    .merge(all_slots, on="time_slot", how="left")
)

# Average taxi-out time per runway and 30-minute slot over all days.
time_stats = (
    plot_df
    .groupby(["runway", "time_slot", "time_label"], as_index=False)
    .agg(
        avg_taxi_time_min=("taxi_time_min", "mean"),
        total_movements=("taxi_time_min", "size"),
    )
)

time_stats = all_runway_slots.merge(
    time_stats,
    on=[
        "runway",
        "time_slot",
        "time_label",
    ],
    how="left",
)

# Number of movements per runway, date, and 30-minute slot.
daily_slot_stats = (
    plot_df
    .groupby(["runway", "date_lt", "time_slot", "time_label"], as_index=False)
    .agg(
        movements=("taxi_time_min", "size"),
    )
)

# Add the daily movement count of the corresponding runway and 30-minute slot to each flight.
plot_df = plot_df.merge(
    daily_slot_stats[
        [
            "runway",
            "date_lt",
            "time_slot",
            "movements",
        ]
    ],
    on=[
        "runway",
        "date_lt",
        "time_slot",
    ],
    how="left",
)


# ========================= Figure =========================

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=False,
    vertical_spacing=0.17,
    row_heights=[0.9, 0.9],
    subplot_titles=(
        "Average Taxi-Out Time by Time of Day and Take-Off Runway",
        "Taxi-Out Time by Number of Movements and Take-Off Runway",
    ),
)

# -------- Row 1: Average taxi-out time per 30-minute slot --------

for runway in RUNWAYS:
    runway_stats = time_stats[time_stats["runway"] == runway]

    fig.add_trace(
        go.Scatter(
            x=runway_stats["time_slot"],
            y=runway_stats["avg_taxi_time_min"],
            mode="lines+markers",
            name=runway,
            legendgroup=runway,
            showlegend=True,
            line=dict(
                color=runway_colours[runway],
                width=LINE_WIDTH,
            ),
            marker=dict(
                size=LINE_MARKER_SIZE,
                color=runway_colours[runway],
            ),
            customdata=runway_stats[
                [
                    "time_label",
                    "total_movements",
                ]
            ].to_numpy(),
            hovertemplate=(
                f"Runway: {runway}<br>"
                "Time: %{customdata[0]}<br>"
                "Average taxi-out time: %{y:.2f} min<br>"
                "Total movements: %{customdata[1]:.0f}"
                "<extra></extra>"
            ),
        ),
        row=1,
        col=1,
    )

# -------- Row 2: Individual taxi-out times by daily movement count --------

for runway in RUNWAYS:
    runway_data = plot_df[plot_df["runway"] == runway]

    fig.add_trace(
        go.Scatter(
            x=runway_data["movements"],
            y=runway_data["taxi_time_min"],
            mode="markers",
            name=runway,
            legendgroup=runway,
            showlegend=False,
            marker=dict(
                size=MARKER_SIZE,
                opacity=MARKER_OPACITY,
                color=runway_colours[runway],
                line=dict(
                    width=0,
                ),
            ),
            customdata=runway_data[
                [
                    "date_lt",
                    "time_label",
                    "time_of_day_label",
                    "taxi_start_lt_label",
                ]
            ].to_numpy(),
            hovertemplate=(
                f"Runway: {runway}<br>"
                "Date: %{customdata[0]}<br>"
                "Time slot: %{customdata[1]}<br>"
                "Local time: %{customdata[2]}<br>"
                "Movements in slot: %{x:.0f}<br>"
                "Taxi-out time: %{y:.2f} min"
                "<extra></extra>"
            ),
        ),
        row=2,
        col=1,
    )

# X-axis ticks from 00:00 to 24:00 for the upper subplot.
tickvals = list(range(0, 48 + 1, TIME_TICK_STEP_HOURS * 2))
ticktext = [f"{h:02d}:00" for h in range(0, 25, TIME_TICK_STEP_HOURS)]

fig.update_layout(
    title=dict(
        text="Taxi-Out Time by Time of Day, Movements, and Take-Off Runway",
        x=0.5,
        xanchor="center",
        y=0.98,
        yanchor="top",
    ),
    title_font=dict(
        size=title_size,
        family=font_family,
        color=font_colour,
    ),
    width=figure_width,
    height=figure_height,
    margin=dict(t=150, b=100, l=100),
    font=dict(
        family=font_family,
        color=font_colour,
    ),
    legend=dict(
        title="Take-Off Runway",
        x=1.02,
        y=0.96,
        xanchor="left",
        yanchor="top",
        font=dict(
            size=legend_font_size,
            family=font_family,
            color=font_colour,
        ),
        title_font=dict(
            size=legend_title_size,
            family=font_family,
            color=font_colour,
        ),
    ),
)

# Upper x-axis: time of day.
fig.update_xaxes(
    title_text="Time of Day (Local Time)",
    range=[0, 48],
    tickmode="array",
    tickvals=tickvals,
    ticktext=ticktext,
    tickangle=90,
    showgrid=True,
    row=1,
    col=1,
)

# Lower x-axis: movements per runway, date, and 30-minute slot.
lower_xaxis_settings = dict(
    title_text="Number of Movements per Runway and 30-Minute Slot",
    showgrid=True,
    row=2,
    col=1,
)

if MOVEMENT_MAX is not None:
    lower_xaxis_settings["range"] = [MOVEMENT_MIN, MOVEMENT_MAX]

fig.update_xaxes(**lower_xaxis_settings)

# Y-axes.
fig.update_yaxes(
    title_text="Average Taxi-Out Time [min]",
    range=[TAXI_TIME_MIN, TAXI_TIME_MAX],
    showgrid=True,
    row=1,
    col=1,
)

fig.update_yaxes(
    title_text="Taxi-Out Time [min]",
    range=[TAXI_TIME_MIN, TAXI_TIME_MAX],
    showgrid=True,
    row=2,
    col=1,
)

# Global axis font sizes and styles.
fig.update_xaxes(
    title_font=dict(size=axis_title_size, family=font_family, color=font_colour),
    tickfont=dict(size=axis_tick_size, family=font_family, color=font_colour),
)

fig.update_yaxes(
    title_font=dict(size=axis_title_size, family=font_family, color=font_colour),
    tickfont=dict(size=axis_tick_size, family=font_family, color=font_colour),
)

# Subplot title fonts.
fig.update_annotations(
    font=dict(size=subplot_title_size, family=font_family, color=font_colour)
)

fig.show()
fig.write_image("figures/taxi_out_time_by_time_of_day_and_movements.pdf")

In [7]:
# ========================= Adjust the parameters here inside =========================

RUNWAYS = ["28", "32", "16", "10", "34"]  # Runway order in the plot.

SCATTER_RUNWAYS = ["28"]                  # Runway(s) shown in the lower scatter plot.

TAXI_TIME_MIN = -1                        # [min] Lower y-axis limit.
TAXI_TIME_MAX = 46                        # [min] Upper y-axis limit.

MOVEMENT_MIN = 0                          # Lower x-axis limit for movements.
MOVEMENT_MAX = None                       # Upper x-axis limit for movements. Use None for automatic scaling.

TIME_TICK_STEP_HOURS = 1                  # X-axis tick spacing in hours.

MARKER_SIZE = 7                           # Scatter marker size in the plot.
LEGEND_MARKER_SIZE = 11                   # Scatter marker size in the legend.
MARKER_OPACITY = 0.65                     # Scatter marker opacity.

LINE_WIDTH = 3                            # Line width in the upper subplot.
LINE_MARKER_SIZE = 8                      # Marker size for the line plot.

# --- Parameters for fonts ---
font_family = "Arial"
font_colour = "black"

title_size = 35
axis_title_size = 25
axis_tick_size = 25
legend_title_size = 30
legend_font_size = 25
subplot_title_size = 30

# --- Figure size ---
figure_width = 1500
figure_height = 1200

# --- Colours ---
runway_colours = {
    "28": "royalblue",
    "32": "darkred",
    "16": "darkorange",
    "10": "darkgreen",
    "34": "darkviolet",
}

# ========================= Adjust the parameters here inside =========================


df = features_df.copy()

# Convert taxi start time from UTC to Swiss local time.
df["taxi_start"] = pd.to_datetime(df["taxi_start"], utc=True)
df["taxi_start_lt"] = df["taxi_start"].dt.tz_convert("Europe/Zurich")

# Extract the local date.
df["date_lt"] = df["taxi_start_lt"].dt.date

# Keep only rows with valid taxi start times and taxi-out times.
df = df.dropna(
    subset=[
        "taxi_start_lt",
        "taxi_time_min",
    ]
)

# Keep only rows with non-negative taxi times.
df = df[df["taxi_time_min"] >= 0]

# Remove taxi-time outliers outside the plotted y-axis range.
df = df[
    (df["taxi_time_min"] >= TAXI_TIME_MIN)
    & (df["taxi_time_min"] <= TAXI_TIME_MAX)
].copy()

# Convert local time of day to 30-minute slots.
df["time_slot"] = (
    df["taxi_start_lt"].dt.hour * 60
    + df["taxi_start_lt"].dt.minute
) // 30


def slot_label(slot: int) -> str:
    """Return a time interval label for a given 30-minute slot index."""
    start_min = slot * 30
    end_min = start_min + 29
    h1, m1 = divmod(start_min, 60)
    h2, m2 = divmod(end_min, 60)
    return f"{h1:02d}:{m1:02d}–{h2:02d}:{m2:02d}"


# Create local-time labels for hover information.
df["time_label"] = df["time_slot"].apply(slot_label)
df["time_of_day_label"] = df["taxi_start_lt"].dt.strftime("%H:%M:%S")
df["taxi_start_lt_label"] = df["taxi_start_lt"].dt.strftime("%Y-%m-%d %H:%M:%S %Z")


# ========================= Prepare data =========================

runway_dfs = []

for runway in RUNWAYS:
    runway_col = f"TO_runway_{runway}"

    tmp = df[df[runway_col] == 1].copy()
    tmp["runway"] = runway

    runway_dfs.append(tmp)

plot_df = pd.concat(runway_dfs, ignore_index=True)

# Define the full ordered list of 30-minute slots.
time_order = [slot_label(i) for i in range(48)]

all_slots = pd.DataFrame({
    "time_slot": range(48),
    "time_label": time_order,
})

all_runway_slots = (
    pd.MultiIndex
    .from_product([RUNWAYS, range(48)], names=["runway", "time_slot"])
    .to_frame(index=False)
    .merge(all_slots, on="time_slot", how="left")
)

# Average taxi-out time per runway and 30-minute slot over all days.
time_stats = (
    plot_df
    .groupby(["runway", "time_slot", "time_label"], as_index=False)
    .agg(
        avg_taxi_time_min=("taxi_time_min", "mean"),
        total_movements=("taxi_time_min", "size"),
    )
)

time_stats = all_runway_slots.merge(
    time_stats,
    on=[
        "runway",
        "time_slot",
        "time_label",
    ],
    how="left",
)

# Number of movements per runway, date, and 30-minute slot.
daily_slot_stats = (
    plot_df
    .groupby(["runway", "date_lt", "time_slot", "time_label"], as_index=False)
    .agg(
        movements=("taxi_time_min", "size"),
    )
)

# Add the daily movement count of the corresponding runway and 30-minute slot to each flight.
plot_df = plot_df.merge(
    daily_slot_stats[
        [
            "runway",
            "date_lt",
            "time_slot",
            "movements",
        ]
    ],
    on=[
        "runway",
        "date_lt",
        "time_slot",
    ],
    how="left",
)


# ========================= Figure =========================

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=False,
    vertical_spacing=0.17,
    row_heights=[0.9, 0.9],
    subplot_titles=(
        "Average Taxi-Out Time by Time of Day and Take-Off Runway",
        "Taxi-Out Time by Number of Movements and Take-Off Runway",
    ),
)

# -------- Row 1: Average taxi-out time per 30-minute slot --------

for runway in RUNWAYS:
    runway_stats = time_stats[time_stats["runway"] == runway]

    fig.add_trace(
        go.Scatter(
            x=runway_stats["time_slot"],
            y=runway_stats["avg_taxi_time_min"],
            mode="lines+markers",
            name=runway,
            legendgroup=runway,
            showlegend=True,
            line=dict(
                color=runway_colours[runway],
                width=LINE_WIDTH,
            ),
            marker=dict(
                size=LINE_MARKER_SIZE,
                color=runway_colours[runway],
            ),
            customdata=runway_stats[
                [
                    "time_label",
                    "total_movements",
                ]
            ].to_numpy(),
            hovertemplate=(
                f"Runway: {runway}<br>"
                "Time: %{customdata[0]}<br>"
                "Average taxi-out time: %{y:.2f} min<br>"
                "Total movements: %{customdata[1]:.0f}"
                "<extra></extra>"
            ),
        ),
        row=1,
        col=1,
    )

# -------- Row 2: Individual taxi-out times by daily movement count --------

for runway in SCATTER_RUNWAYS:
    runway_data = plot_df[plot_df["runway"] == runway]

    fig.add_trace(
        go.Scatter(
            x=runway_data["movements"],
            y=runway_data["taxi_time_min"],
            mode="markers",
            name=runway,
            legendgroup=runway,
            showlegend=False,
            marker=dict(
                size=MARKER_SIZE,
                opacity=MARKER_OPACITY,
                color=runway_colours[runway],
                line=dict(
                    width=0,
                ),
            ),
            customdata=runway_data[
                [
                    "date_lt",
                    "time_label",
                    "time_of_day_label",
                    "taxi_start_lt_label",
                ]
            ].to_numpy(),
            hovertemplate=(
                f"Runway: {runway}<br>"
                "Date: %{customdata[0]}<br>"
                "Time slot: %{customdata[1]}<br>"
                "Local time: %{customdata[2]}<br>"
                "Movements in slot: %{x:.0f}<br>"
                "Taxi-out time: %{y:.2f} min"
                "<extra></extra>"
            ),
        ),
        row=2,
        col=1,
    )

# X-axis ticks from 00:00 to 24:00 for the upper subplot.
tickvals = list(range(0, 48 + 1, TIME_TICK_STEP_HOURS * 2))
ticktext = [f"{h:02d}:00" for h in range(0, 25, TIME_TICK_STEP_HOURS)]

fig.update_layout(
    title=dict(
        text="Taxi-Out Time by Time of Day, Movements, and Take-Off Runway",
        x=0.5,
        xanchor="center",
        y=0.98,
        yanchor="top",
    ),
    title_font=dict(
        size=title_size,
        family=font_family,
        color=font_colour,
    ),
    width=figure_width,
    height=figure_height,
    margin=dict(t=150, b=100, l=100),
    font=dict(
        family=font_family,
        color=font_colour,
    ),
    legend=dict(
        title="Take-Off Runway",
        x=1.02,
        y=0.96,
        xanchor="left",
        yanchor="top",
        font=dict(
            size=legend_font_size,
            family=font_family,
            color=font_colour,
        ),
        title_font=dict(
            size=legend_title_size,
            family=font_family,
            color=font_colour,
        ),
    ),
)

# Upper x-axis: time of day.
fig.update_xaxes(
    title_text="Time of Day (Local Time)",
    range=[0, 48],
    tickmode="array",
    tickvals=tickvals,
    ticktext=ticktext,
    tickangle=90,
    showgrid=True,
    row=1,
    col=1,
)

# Lower x-axis: movements per runway, date, and 30-minute slot.
lower_xaxis_settings = dict(
    title_text="Number of Movements per Runway and 30-Minute Slot",
    showgrid=True,
    row=2,
    col=1,
)

if MOVEMENT_MAX is not None:
    lower_xaxis_settings["range"] = [MOVEMENT_MIN, MOVEMENT_MAX]

fig.update_xaxes(**lower_xaxis_settings)

# Y-axes.
fig.update_yaxes(
    title_text="Average Taxi-Out Time [min]",
    range=[TAXI_TIME_MIN, TAXI_TIME_MAX],
    showgrid=True,
    row=1,
    col=1,
)

fig.update_yaxes(
    title_text="Taxi-Out Time [min]",
    range=[TAXI_TIME_MIN, TAXI_TIME_MAX],
    showgrid=True,
    row=2,
    col=1,
)

# Global axis font sizes and styles.
fig.update_xaxes(
    title_font=dict(size=axis_title_size, family=font_family, color=font_colour),
    tickfont=dict(size=axis_tick_size, family=font_family, color=font_colour),
)

fig.update_yaxes(
    title_font=dict(size=axis_title_size, family=font_family, color=font_colour),
    tickfont=dict(size=axis_tick_size, family=font_family, color=font_colour),
)

# Subplot title fonts.
fig.update_annotations(
    font=dict(size=subplot_title_size, family=font_family, color=font_colour)
)

fig.show()

runway_string = "_".join(SCATTER_RUNWAYS)
runway_label = "runway" if len(SCATTER_RUNWAYS) == 1 else "runways"

fig.write_image(f"figures/taxi_out_time_by_time_of_day_and_movements_{runway_label}_{runway_string}.pdf")

In [8]:
# ========================= Adjust the parameters here inside =========================

RUNWAYS = ["28", "32", "16", "10", "34"]  # Runway order in the plot.

TAXI_TIME_MIN = -1                        # [min] Lower y-axis limit.
TAXI_TIME_MAX = 46                        # [min] Upper y-axis limit.

TIME_TICK_STEP_HOURS = 1                  # X-axis tick spacing in hours.

LINE_WIDTH = 4                            # Line width in the plot.
MARKER_SIZE = 9                           # Marker size in the plot.

# --- Parameters for fonts ---
font_family = "Arial"
font_colour = "black"

title_size = 35
axis_title_size = 25
axis_tick_size = 25
legend_title_size = 30
legend_font_size = 25

# --- Figure size ---
figure_width = 1500
figure_height = 900

# --- Colours ---
runway_colours = {
    "28": "royalblue",
    "32": "darkred",
    "16": "darkorange",
    "10": "darkgreen",
    "34": "darkviolet",
}

# ========================= Adjust the parameters here inside =========================


df = features_df.copy()

# Convert taxi start time from UTC to Swiss local time.
df["taxi_start"] = pd.to_datetime(df["taxi_start"], utc=True)
df["taxi_start_lt"] = df["taxi_start"].dt.tz_convert("Europe/Zurich")

# Keep only rows with valid taxi start times and taxi-out times.
df = df.dropna(
    subset=[
        "taxi_start_lt",
        "taxi_time_min",
    ]
)

# Keep only rows with non-negative taxi times.
df = df[df["taxi_time_min"] >= 0]

# Remove taxi-time outliers outside the plotted y-axis range.
df = df[
    (df["taxi_time_min"] >= TAXI_TIME_MIN)
    & (df["taxi_time_min"] <= TAXI_TIME_MAX)
].copy()

# Convert local time of day to 30-minute slots.
df["time_slot"] = (
    df["taxi_start_lt"].dt.hour * 60
    + df["taxi_start_lt"].dt.minute
) // 30


def slot_label(slot: int) -> str:
    """Return a time interval label for a given 30-minute slot index."""
    start_min = slot * 30
    end_min = start_min + 29
    h1, m1 = divmod(start_min, 60)
    h2, m2 = divmod(end_min, 60)
    return f"{h1:02d}:{m1:02d}–{h2:02d}:{m2:02d}"


# Create local-time labels for hover information.
df["time_label"] = df["time_slot"].apply(slot_label)


# ========================= Prepare data =========================

runway_dfs = []

for runway in RUNWAYS:
    runway_col = f"TO_runway_{runway}"

    tmp = df[df[runway_col] == 1].copy()
    tmp["runway"] = runway

    runway_dfs.append(tmp)

plot_df = pd.concat(runway_dfs, ignore_index=True)

# Define the full ordered list of 30-minute slots.
time_order = [slot_label(i) for i in range(48)]

all_slots = pd.DataFrame({
    "time_slot": range(48),
    "time_label": time_order,
})

all_runway_slots = (
    pd.MultiIndex
    .from_product([RUNWAYS, range(48)], names=["runway", "time_slot"])
    .to_frame(index=False)
    .merge(all_slots, on="time_slot", how="left")
)

# Calculate the average taxi-out time per runway and 30-minute slot over all days.
time_stats = (
    plot_df
    .groupby(["runway", "time_slot", "time_label"], as_index=False)
    .agg(
        avg_taxi_time_min=("taxi_time_min", "mean"),
        movements=("taxi_time_min", "size"),
    )
)

time_stats = all_runway_slots.merge(
    time_stats,
    on=[
        "runway",
        "time_slot",
        "time_label",
    ],
    how="left",
)


# ========================= Figure =========================

fig = go.Figure()

# Add one line trace per runway.
for runway in RUNWAYS:
    runway_stats = time_stats[time_stats["runway"] == runway]

    fig.add_trace(
        go.Scatter(
            x=runway_stats["time_slot"],
            y=runway_stats["avg_taxi_time_min"],
            mode="lines+markers",
            name=runway,
            line=dict(
                color=runway_colours[runway],
                width=LINE_WIDTH,
            ),
            marker=dict(
                size=MARKER_SIZE,
                color=runway_colours[runway],
            ),
            customdata=runway_stats[
                [
                    "time_label",
                    "movements",
                ]
            ].to_numpy(),
            hovertemplate=(
                f"Runway: {runway}<br>"
                "Time: %{customdata[0]}<br>"
                "Average taxi-out time: %{y:.2f} min<br>"
                "Total movements: %{customdata[1]:.0f}"
                "<extra></extra>"
            ),
        )
    )

# X-axis ticks from 00:00 to 24:00.
tickvals = list(range(0, 48 + 1, TIME_TICK_STEP_HOURS * 2))
ticktext = [f"{h:02d}:00" for h in range(0, 25, TIME_TICK_STEP_HOURS)]

fig.update_layout(
    title=dict(
        text="Average Taxi-Out Time by Time of Day and Take-Off Runway",
        x=0.5,
        xanchor="center",
        y=0.97,
        yanchor="top",
    ),
    title_font=dict(
        size=title_size,
        family=font_family,
        color=font_colour,
    ),
    width=figure_width,
    height=figure_height,
    margin=dict(t=120, b=100, l=100),
    font=dict(
        family=font_family,
        color=font_colour,
    ),
    legend=dict(
        title="Take-Off Runway",
        x=0.986,
        y=0.974,
        xanchor="right",
        yanchor="top",
        font=dict(
            size=legend_font_size,
            family=font_family,
            color=font_colour,
        ),
        title_font=dict(
            size=legend_title_size,
            family=font_family,
            color=font_colour,
        ),
    ),
    xaxis=dict(
        title="Time of Day (Local Time)",
        range=[0, 48],
        tickmode="array",
        tickvals=tickvals,
        ticktext=ticktext,
        tickangle=90,
        title_font=dict(
            size=axis_title_size,
            family=font_family,
            color=font_colour,
        ),
        tickfont=dict(
            size=axis_tick_size,
            family=font_family,
            color=font_colour,
        ),
        showgrid=True,
    ),
    yaxis=dict(
        title="Average Taxi-Out Time [min]",
        range=[TAXI_TIME_MIN, TAXI_TIME_MAX],
        title_font=dict(
            size=axis_title_size,
            family=font_family,
            color=font_colour,
        ),
        tickfont=dict(
            size=axis_tick_size,
            family=font_family,
            color=font_colour,
        ),
        showgrid=True,
    ),
)

fig.show()
fig.write_image("figures/taxi_out_time_average_by_time_of_day.pdf")